# Projet 9 — Traitement Big Data sur le Cloud

Ce notebook valide la chaîne **image → ResNet50 → PCA → Parquet** avant son exécution distribuée sur AWS EMR. Aucun classificateur n'est entraîné.

## 1. Préparation de Colab

Cette cellule récupère le dépôt et installe uniquement les dépendances absentes de Colab.

In [ ]:
# Prépare un environnement Colab reproductible à partir du dépôt GitHub.
# Les installations restent limitées aux dépendances nécessaires au projet.
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/projet9-big-data-fruits')
if not PROJECT_ROOT.exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/neutrinoox/projet9-big-data-fruits.git',
        str(PROJECT_ROOT),
    ], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT, check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pyspark>=3.5,<4.0', 'pyarrow>=14,<19', 'kagglehub>=1,<2',
], check=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Environnement prêt :', PROJECT_ROOT)

## 2. Téléchargement et échantillon

Fruits-360 est téléchargé depuis Kaggle, puis un échantillon équilibré de 10 classes et 100 images est préparé pour la preuve de concept.

In [ ]:
# Télécharge le dataset public Fruits-360 puis construit un échantillon équilibré.
# L'échantillon accélère les validations locales sans modifier les données sources.
subprocess.run([sys.executable, '-m', 'scripts.download_dataset'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_sample'], check=True)

## 3. Imports et paramètres

Le test utilise un petit échantillon pour limiter la mémoire et le temps de calcul.

In [ ]:
# Importe les fonctions du projet et centralise les paramètres de l'expérience.
# Les mêmes chemins seront remplacés par des URI S3 lors du passage sur AWS.
import pandas as pd
import matplotlib.pyplot as plt
from src.pipeline_local import run_local_pipeline
from src.validate_dataset import find_images

INPUT_DIR = PROJECT_ROOT / 'data' / 'sample'
OUTPUT_FILE = PROJECT_ROOT / 'outputs' / 'local_features_pca.parquet'
MAX_IMAGES = 100
N_COMPONENTS = 20

## 4. Contrôle du dataset

In [ ]:
# Vérifie le volume de données et la répartition des classes avant le traitement.
# Ce contrôle évite de lancer ResNet50 sur un échantillon vide ou déséquilibré.
image_paths = find_images(INPUT_DIR)
labels = pd.Series([path.parent.name for path in image_paths], name='label')
print(f'{len(image_paths)} images — {labels.nunique()} classes')
labels.value_counts().head(10)

## 5. Extraction et réduction

ResNet50, pré-entraîné sur ImageNet, transforme chaque image en 2 048 caractéristiques. La PCA condense ensuite ces vecteurs.

In [ ]:
# Extrait 2 048 caractéristiques par image avec ResNet50, puis applique une PCA.
# Le résultat réduit est enregistré en Parquet pour préparer le traitement distribué.
result, pca = run_local_pipeline(
    input_path=INPUT_DIR,
    output_path=OUTPUT_FILE,
    max_images=MAX_IMAGES,
    components=N_COMPONENTS,
    batch_size=16,
)
print(f'Forme de la sortie : {result.shape}')
print(f'Variance expliquée cumulée : {pca.explained_variance_ratio_.sum():.2%}')
result.head()

## 6. Choix du nombre de composantes

In [ ]:
# Visualise la variance cumulée afin de justifier le nombre de composantes retenu.
# La ligne à 90 % fournit un seuil de comparaison simple pour la soutenance.
cumulative_variance = pca.explained_variance_ratio_.cumsum()
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o')
plt.axhline(0.90, color='red', linestyle='--', label='90 %')
plt.xlabel('Nombre de composantes')
plt.ylabel('Variance expliquée cumulée')
plt.title('Information conservée par la PCA')
plt.grid(alpha=0.3)
plt.legend();

## 7. Contrôle du fichier Parquet

Le format Parquet est compact, typé et adapté aux traitements distribués.

In [ ]:
# Relit le fichier Parquet et contrôle sa cohérence avec le résultat en mémoire.
# Les assertions interrompent le notebook si la sauvegarde est incomplète.
saved = pd.read_parquet(OUTPUT_FILE)
assert len(saved) == len(result)
assert saved['pca_features'].map(len).nunique() == 1
print('Parquet valide :', OUTPUT_FILE)
saved.head()

## 8. Exécution distribuée avec Spark

Cette étape exécute la chaîne complète avec lecture récursive, traitement par partitions, diffusion des poids ResNet50 par broadcast et PCA Spark.

In [ ]:
# Lance le pipeline Spark sur le même échantillon que la preuve de concept locale.
# Le script affiche le nombre d'images, les dimensions et la variance conservée.
subprocess.run([
    sys.executable, '-m', 'src.pipeline_spark',
    '--input', 'data/sample',
    '--output', 'outputs/spark_pca',
    '--max-images', str(MAX_IMAGES),
    '--components', str(N_COMPONENTS),
], cwd=PROJECT_ROOT, check=True)

## 9. Contrôle de la sortie Spark

In [ ]:
# Vérifie que Spark a produit au moins un fragment Parquet et son fichier de succès.
# Cette vérification confirme l'écriture distribuée avant le passage vers S3.
SPARK_OUTPUT = PROJECT_ROOT / 'outputs' / 'spark_pca'
parquet_parts = list(SPARK_OUTPUT.glob('part-*.parquet'))
assert parquet_parts, 'Aucun fragment Parquet produit par Spark.'
assert (SPARK_OUTPUT / '_SUCCESS').exists(), 'Écriture Spark incomplète.'
print(f'Sortie Spark valide : {len(parquet_parts)} fragment(s) Parquet')

## Conclusion

Les preuves de concept locale et Spark valident la chaîne complète sur un échantillon équilibré. L'étape suivante sera son exécution sur S3 et EMR, sans modifier la logique de traitement.